## CHEAP/RICH Analysis on the BTP Curve:


- Curve fitting using NS
- Cheap/Rich based on 1m Z score of model's residuals


In [1]:
import datetime as dt
import pandas as pd
import numpy as np

from datetime import datetime, date, time, timedelta
from dateutil.relativedelta import relativedelta
from scipy.optimize import least_squares

#### User defined functions

In [29]:
def creating_database(df_bdp, df_bdh):
    res = pd.DataFrame()
    for col in list(df_bdh.columns):
        for ind in range(len(df_bdh.index)):
            if col in df_bdp.index:
                date = df_bdh.index[ind]
                cusip = col
                yield_= df_bdh[col].iloc[ind]
                mat_date = df_bdp.loc[col]['MATURITY']
                yrs_to_mat = df_bdp.loc[col]['MTY_YRS']
                sec_name = df_bdp.loc[col]['NAME']
                tmp = pd.DataFrame([date, cusip, yield_, mat_date, yrs_to_mat, sec_name])
                res = pd.concat([res, tmp], 1)
    
    res = res.T
    cols_name = ['Date', 'Cusip', 'YTM', 'Mat_Date', 'Yrs_to_mat', 'NAME']
    res.columns = cols_name
    res = res.sort_values(by = 'Date')
    res = res.set_index('Date')
    return res.dropna()

def NS_helper(time_, beta_, lamda= 0.0609):
    level_factor = 1
    slope_factor = (1-np.exp((-1)*lamda*time_))/(lamda*time_)
    curvature_factor =(1-np.exp((-1)*lamda*time_))/(lamda*time_)-np.exp((-1)*lamda*time_)
    return(beta_[0]*level_factor + beta_[1]*slope_factor + beta_[2]*curvature_factor)

def NSS_yield(time_list, beta_list):
    yields = []
    for t in time_list:
        yields.append(NS_helper(t, beta_list))
    return yields

def factor_loading(time_, lamda = 0.0609):
    level_factor_loading =[]
    slope_factor_loading = []
    curvature_factor_loading = []
    for i in time_:
        level_factor_loading.append(1)
        slope_factor_loading.append((1-np.exp((-1)*lamda*i))/(lamda*i))
        curvature_factor_loading.append((1 -np.exp((-1)*lamda*i))/(lamda*i) -np.exp((-1)*lamda*i))
    return level_factor_loading, slope_factor_loading, curvature_factor_loading


def residual(beta_, time_array, yields_):
    errors =[]
    c_index = 0
    for i in yields_:
        errors.append(i - NS_helper(time_array[c_index], beta_))
        c_index += 1
    return np.array(errors)


def NS_calibration_routine(df, lamda = 0.0609, gen_points = 1000, b0 =1, 
                          b1 =1, b2 =2):
    time_points = list(df['Yrs_to_mat'])
    ytm_points = list(df['YTM'])
    curve_points = np.linspace(min(time_points), max(time_points),gen_points)
    beta =[b0, b1, b2]
    
    b0_loading, b1_loading, b2_loading = factor_loading(curve_points)
    res = least_squares(residual, beta, args =(time_points, ytm_points))
    opt_params = res.x
    residuals_final = residual(opt_params, time_points, ytm_points)
    res_df = pd.DataFrame(residuals_final, index = df.Cusip)
    res_df.columns = [df.index[0]]
    return res_df.T, opt_params, time_points

def NS_yield_routine(df, lamda = 0.0609):
    ind = list(df.Cusip)
    res, params, time = NS_calibration_routine(df, lamda)
    yield_estim = pd.DataFrame(NSS_yield(time, params), index = ind)
    yield_estim.columns = [df.index[0]]
    return yield_estim.T

def zscore_using_apply(x, window):
    def zscore_func(x):
        return (x[-1]- x[:-1].mean())/x[:-1].std(ddof =0)
    return x.rolling(window = window +1).apply(zscore_func, raw = True)
    
    

### Import from Excel:

In [6]:
sheets_name = ['IT_des', 'IT_yields']

rows_to_skip_des = range(1, 12)
rows_to_skip_hist = range(1, 4)

xl_name = 'BTPs_Splines_V3.xlsx'


#### Descriptive Database:

In [4]:
des = pd.read_excel(xl_name, sheet_name= sheets_name[0], 
                   skiprows= rows_to_skip_des, header = 1).dropna(axis =1).iloc[1:].set_index('ID')

des = des.rename(columns = {'#Years_Maturity': "MTY_YRS", '#concat': "NAME"})


In [5]:
des

,ID_ISIN,MATURITY,CNTRY_OF_DOMICILE,MTY_YRS,NAME
ID,,,,,
AX870340 Corp,IT0005367492,2024-07-01,IT,2.236824,BTPS 1.75 2024-07-01
BO383338 Corp,IT0005438004,2045-04-30,IT,23.066393,BTPS 1.5 2045-04-30
ZO167103 Corp,IT0005419848,2026-02-01,IT,3.824778,BTPS 0.5 2026-02-01
EC065947 Corp,IT0001278511,2029-11-01,IT,7.572895,BTPS 5.25 2029-11-01
BN471867 Corp,IT0005433690,2028-03-15,IT,5.941136,BTPS 0.25 2028-03-15
...,...,...,...,...,...
AX099966 Corp,IT0005363111,2049-09-01,IT,27.405886,BTPS 3.85 2049-09-01
BG013029 Corp,IT0005402117,2036-03-01,IT,13.902806,BTPS 1.45 2036-03-01
BK459727 Corp,IT0005416570,2027-09-15,IT,5.442847,BTPS 0.95 2027-09-15


#### Yield Database

In [8]:
ytm_BTP = pd.read_excel(xl_name, sheet_name= sheets_name[-1], 
                       skiprows= rows_to_skip_hist, header = 1).dropna(axis = 1).iloc[1:]

In [10]:
ytm_BTP.head()

,Unnamed: 0,BS132409 Corp,AX870340 Corp,BO383338 Corp,ZO167103 Corp,EC065947 Corp,BN471867 Corp,AM293682 Corp,EK769119 Corp,EH896068 Corp,...,UV656956 Corp,AP093098 Corp,EC237370 Corp,EF130795 Corp,QZ016755 Corp,AW703520 Corp,AX099966 Corp,BG013029 Corp,BK459727 Corp,JK840318 Corp
1,2022-04-05 00:00:00,2.26014,0.728854,2.56001,1.19082,1.76945,1.70322,1.51613,0.997378,0.901101,...,1.12725,0.898011,2.018,2.38909,1.37841,2.35228,2.6781,2.34744,1.58119,2.36586
2,2022-04-04 00:00:00,2.06767,0.605717,2.39562,1.04629,1.5875,1.53959,1.35816,0.8655,0.767892,...,0.984758,0.763,1.82604,2.21321,1.22302,2.17806,2.50244,2.17398,1.4222,2.19184
3,2022-04-01 00:00:00,2.09064,0.611988,2.41534,1.05976,1.59021,1.55999,1.3796,0.878174,0.774821,...,0.997861,0.775264,1.85101,2.24239,1.24264,2.20677,2.5178,2.20519,1.4433,2.22192
4,2022-03-31 00:00:00,2.03687,0.573878,2.34738,1.01486,1.52855,1.50822,1.32775,0.835025,0.729645,...,0.951129,0.731759,1.79771,2.18061,1.19369,2.14448,2.44941,2.1459,1.39206,2.15828
5,2022-03-30 00:00:00,2.12656,0.65465,2.40744,1.10647,1.62188,1.59716,1.41693,0.913412,0.808594,...,1.04246,0.799546,1.89005,2.24861,1.28743,2.22054,2.50228,2.21716,1.48247,2.23074


#### Creating Database:

In [24]:
database = creating_database(des, ytm_BTP)

In [25]:
database.head()

,Cusip,YTM,Mat_Date,Yrs_to_mat,NAME
Date,,,,,
1,AX870340 Corp,0.728854,2024-07-01,2.23682,BTPS 1.75 2024-07-01
1,EJ514272 Corp,1.71996,2028-09-01,6.40657,BTPS 4.75 2028-09-01
1,AZ128673 Corp,2.51523,2040-03-01,17.9028,BTPS 3.1 2040-03-01
1,EF130795 Corp,2.38909,2037-02-01,14.8255,BTPS 4.0 2037-02-01
1,EH896068 Corp,0.901101,2025-03-01,2.90212,BTPS 5.0 2025-03-01


#### Computing Residuals from NS Model:

In [27]:
inter_index = list(database.index.drop_duplicates())

In [30]:
hist_res = pd.DataFrame()
for i in inter_index:
    tmp = database[database.index == i].sort_values(by ='Yrs_to_mat')
    df_hist, params_hist, tenors = NS_calibration_routine(tmp)
    hist_res = pd.concat([hist_res, df_hist], 0)

##### Residuals at Bond level

In [32]:
hist_res.head(10)

Cusip,BO470172 Corp,AM796312 Corp,AX870340 Corp,BQ468013 Corp,EK093352 Corp,AP093098 Corp,EK457296 Corp,ZR788824 Corp,EH896068 Corp,AR689537 Corp,...,EJ679466 Corp,BO383338 Corp,EK699416 Corp,JV910526 Corp,AN868354 Corp,AX099966 Corp,ZP513132 Corp,BM080712 Corp,QZ766515 Corp,BO952242 Corp
1,-0.230207,-0.227903,-0.163829,-0.162498,-0.212481,-0.071011,-0.143009,-0.110945,-0.125348,-0.043106,...,-0.159733,-0.195441,-0.169786,-0.178979,-0.158400,-0.123750,-0.098437,-0.159428,0.283920,0.305018
2,-0.215286,-0.216477,-0.156422,-0.153320,-0.204241,-0.071600,-0.136410,-0.102283,-0.121254,-0.042957,...,-0.156721,-0.181780,-0.161567,-0.179459,-0.160755,-0.133508,-0.116142,-0.177666,0.290617,0.296746
3,-0.215920,-0.221199,-0.160682,-0.156277,-0.209409,-0.070842,-0.139930,-0.111840,-0.126554,-0.041550,...,-0.157130,-0.185937,-0.175813,-0.182908,-0.164182,-0.139413,-0.117906,-0.179003,0.290435,0.309118
4,-0.213723,-0.223023,-0.159170,-0.154940,-0.207934,-0.073464,-0.143060,-0.116156,-0.129904,-0.043674,...,-0.157518,-0.188065,-0.173663,-0.185250,-0.176011,-0.142592,-0.126781,-0.185396,0.302302,0.310179
5,-0.230406,-0.234584,-0.163547,-0.170894,-0.220547,-0.091068,-0.151324,-0.125696,-0.136507,-0.057864,...,-0.163984,-0.194188,-0.173502,-0.186610,-0.178674,-0.148403,-0.121247,-0.182287,0.305637,0.323533
6,-0.213860,-0.216585,-0.146239,-0.161128,-0.208179,-0.086624,-0.144679,-0.122167,-0.131970,-0.056631,...,-0.169340,-0.198870,-0.179018,-0.186606,-0.177978,-0.149901,-0.124011,-0.178116,0.300058,0.329965
7,-0.201510,-0.204807,-0.131919,-0.150990,-0.194952,-0.083054,-0.133481,-0.113492,-0.132565,-0.057528,...,-0.171750,-0.201900,-0.177557,-0.184959,-0.175041,-0.144023,-0.126130,-0.179119,0.304432,0.320971
8,-0.187122,-0.189033,-0.116924,-0.140840,-0.182480,-0.072182,-0.121608,-0.108056,-0.128293,-0.060648,...,-0.172125,-0.205135,-0.178621,-0.190102,-0.172989,-0.145871,-0.128516,-0.187122,0.298189,0.326526
9,-0.186272,-0.185653,-0.116447,-0.142902,-0.179289,-0.075378,-0.118633,-0.109760,-0.130577,-0.062045,...,-0.168395,-0.202861,-0.177803,-0.187576,-0.170076,-0.144215,-0.129849,-0.188217,0.293395,0.325911
10,-0.174594,-0.178709,-0.111662,-0.139841,-0.165893,-0.069716,-0.109717,-0.104558,-0.120875,-0.060830,...,-0.165620,-0.199630,-0.173865,-0.185970,-0.169474,-0.144054,-0.127978,-0.191242,0.306831,0.305463


#### Est Yields from model:

In [33]:
est_yields_df = pd.DataFrame()
for i in inter_index:
    tmp = database[database.index == i].sort_values(by = 'Yrs_to_mat')
    df_yield = NS_yield_routine(tmp)
    est_yields_df = pd.concat([est_yields_df, df_yield], 0)
    

In [34]:
est_yields_df.head()

,BO470172 Corp,AM796312 Corp,AX870340 Corp,BQ468013 Corp,EK093352 Corp,AP093098 Corp,EK457296 Corp,ZR788824 Corp,EH896068 Corp,AR689537 Corp,...,EJ679466 Corp,BO383338 Corp,EK699416 Corp,JV910526 Corp,AN868354 Corp,AX099966 Corp,ZP513132 Corp,BM080712 Corp,QZ766515 Corp,BO952242 Corp
1,0.848705,0.865932,0.892682,0.918024,0.927530,0.969022,0.977781,1.011416,1.026448,1.066233,...,2.742737,2.755454,2.776294,2.782432,2.792431,2.801855,2.804758,2.805231,2.619383,2.521396
2,0.720421,0.736761,0.762139,0.786187,0.795209,0.834600,0.842917,0.874865,0.889146,0.926955,...,2.563144,2.577398,2.601638,2.609119,2.621965,2.635944,2.642043,2.645825,2.517706,2.438339
3,0.730387,0.746948,0.772671,0.797043,0.806186,0.846105,0.854534,0.886905,0.901375,0.939682,...,2.587347,2.601281,2.624802,2.631997,2.644229,2.657217,2.662616,2.665667,2.524518,2.440890
4,0.691492,0.707768,0.733048,0.757002,0.765988,0.805223,0.813507,0.845326,0.859549,0.897203,...,2.521509,2.535443,2.559048,2.566300,2.578689,2.592005,2.597676,2.601039,2.467589,2.386887
5,0.776489,0.792826,0.818197,0.842235,0.851251,0.890614,0.898924,0.930837,0.945101,0.982857,...,2.588805,2.601628,2.622948,2.629346,2.639998,2.650688,2.654606,2.656213,2.496713,2.408506


### Z_scores residuals:

In [35]:
## set at 1m rolling
window = 22

z_score_res = zscore_using_apply(hist_res.sort_index(ascending = False), window)
z_score_res = z_score_res.sort_index(ascending = True)

In [39]:
z_score_res.head()

Cusip,BO470172 Corp,AM796312 Corp,AX870340 Corp,BQ468013 Corp,EK093352 Corp,AP093098 Corp,EK457296 Corp,ZR788824 Corp,EH896068 Corp,AR689537 Corp,...,EJ679466 Corp,BO383338 Corp,EK699416 Corp,JV910526 Corp,AN868354 Corp,AX099966 Corp,ZP513132 Corp,BM080712 Corp,QZ766515 Corp,BO952242 Corp
1,-1.867692,-1.761910,-1.801504,-1.250751,-1.517306,-0.123941,-1.486522,-0.909036,-0.907409,2.157419,...,-0.093291,-0.382382,-0.461778,-0.262578,-0.123784,0.688575,1.597489,0.550138,0.243587,0.389398
2,-1.455952,-1.397502,-1.633932,-0.932814,-1.366318,-0.246792,-1.262273,-0.583181,-0.789511,2.503923,...,0.114717,0.711445,0.060974,-0.381521,-0.339020,-0.156637,-0.096848,-0.569598,0.596873,0.194383
3,-1.596857,-1.774004,-1.990119,-1.131749,-1.690682,-0.277579,-1.563381,-1.090888,-1.129712,3.347842,...,0.014477,0.222642,-0.994290,-0.667277,-0.594238,-0.639279,-0.319637,-0.712884,0.658247,0.644125
4,-1.657203,-2.117487,-2.186477,-1.171623,-1.827509,-0.510690,-1.927685,-1.407001,-1.411433,3.798518,...,-0.083149,0.025196,-0.931281,-0.878984,-1.290436,-0.932011,-1.023003,-1.139683,1.179588,0.746618
5,-2.679515,-3.328506,-2.899839,-2.019481,-2.723623,-1.720299,-2.846760,-2.074857,-1.945516,0.607056,...,-0.675082,-0.447829,-1.007086,-1.044232,-1.580863,-1.444354,-0.708054,-1.073849,1.432833,1.219773


### Export to excel

In [43]:
path_des = "/Users/lollo/Desktop/Exodus/BTPs_descriptive.xlsx"

path_residuals = "/Users/lollo/Desktop/Exodus/NS_BTPs_residuals.xlsx"

path_estyields = "/Users/lollo/Desktop/Exodus/NS_yield_Est_BTPs.xlsx"

path_zscore = "/Users/lollo/Desktop/Exodus/Zscores_Resid_BTPs.xlsx"

##### Export Output

In [45]:
des.to_excel(path_des)

hist_res.to_excel(path_residuals)

est_yields_df.to_excel(path_estyields)

z_score_res.to_excel(path_zscore)